<a href="https://colab.research.google.com/github/zombie9088/Pytorch_learning/blob/main/Transfer%20Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


In [3]:
torch.manual_seed(42)

In [4]:
device= torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
df = pd.read_csv("/content/drive/MyDrive/fmnist/fashion-mnist_train.csv")
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
X= df.iloc[:,1:].values
y= df.iloc[:, 0].values

In [7]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [11]:
#transformations

from torchvision.transforms import transforms
custom_transform= transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
])

In [29]:
#custom datset class

from PIL import Image
import numpy as np

class CustomDataset(Dataset):
  def __init__(self,features,labels,transform):
    self.features= features
    self.labels= labels
    self.transform= transform

  def __len__(self):
    return len(self.features)

  def __getitem__(self,idx):
    #resize to 28x28
    image= self.features[idx].reshape(28,28)

    #change to np.unit8
    image= image.astype(np.uint8)

    #change bnw to color
    image= np.stack([image]*3,axis=-1)

    #convert to PIL image
    image= Image.fromarray(image)

    #apply transformation
    image= self.transform(image)

    #return
    return image, torch.tensor(self.labels[idx],dtype=torch.long)

In [30]:
#creating train dataset object

train_dataset = CustomDataset(X_train,y_train,transform=custom_transform)


In [31]:
#create test dataset object
test_dataset= CustomDataset(X_test,y_test,transform=custom_transform)

In [32]:
#create train and test dataloader

train_loader= DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader= DataLoader(test_dataset,batch_size=32,shuffle=False)

In [33]:
import torchvision.models as models

vgg16= models.vgg16(pretrained=True)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [34]:
vgg16

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [35]:
for param in vgg16.features.parameters():
  param.requires_grad= False


In [36]:
vgg16.classifier=nn.Sequential(
    nn.Linear(25088,1024),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(1024,512),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(512,10)
)

In [37]:
vgg16

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [38]:
vgg16= vgg16.to(device)

In [39]:
#epochs and learning rate

epochs= 10
learning_rate = 0.0001

In [40]:

criterion = nn.CrossEntropyLoss()

#optimiser

optimizer= optim.Adam(vgg16.classifier.parameters(),lr=learning_rate)

In [42]:
#training loop

for epoch in range(epochs):

  total_epoch_loss=0
  for batch_features, batch_labels in train_loader:
    batch_features= batch_features.to(device)
    batch_labels= batch_labels.to(device)

    #forward pass
    outputs= vgg16(batch_features)

    #loss
    loss= criterion(outputs,batch_labels)

    optimizer.zero_grad()

    #backprop
    loss.backward()


    #update gradient
    optimizer.step()
    total_epoch_loss= total_epoch_loss+loss.item()

  print(f"Epoch {epoch+1}/{epochs}, Loss: {total_epoch_loss/len(train_loader)}")


Epoch 1/10, Loss: 0.2627576734088361
Epoch 2/10, Loss: 0.19153811062779277
Epoch 3/10, Loss: 0.15018258749259014
Epoch 4/10, Loss: 0.1171277074503402
Epoch 5/10, Loss: 0.09417527068631414
Epoch 6/10, Loss: 0.0763066121355708
Epoch 7/10, Loss: 0.060485837082572595
Epoch 8/10, Loss: 0.051857646514001925
Epoch 9/10, Loss: 0.04462632768469969
Epoch 10/10, Loss: 0.039891972574288954


In [43]:
#set model to eval mode

vgg16.eval()


VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [45]:
#evaluation code on test data

total=0
correct =0

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    batch_features= batch_features.to(device)
    batch_labels= batch_labels.to(device)

    outputs= vgg16(batch_features)

    _,predicted = torch.max(outputs,1)
    total+= batch_labels.shape[0]
    correct+= (predicted==batch_labels).sum().item()

print(correct/total)

0.9256666666666666


In [46]:
#evaluation code on train data

total=0
correct =0

with torch.no_grad():
  for batch_features, batch_labels in train_loader:
    batch_features= batch_features.to(device)
    batch_labels= batch_labels.to(device)

    outputs= vgg16(batch_features)

    _,predicted = torch.max(outputs,1)
    total+= batch_labels.shape[0]
    correct+= (predicted==batch_labels).sum().item()

print(correct/total)

0.9972916666666667
